# CrashVision — Re-entrenamiento M3: Dataset representativo del dominio
### Sistema de Detección Automática de Accidentes de Tránsito

**Proyecto:** CrashVision — Inteligencia Artificial 1  
**Universidad:** UDLA (Universidad de las Américas, Quito)  
**Docente:** Prof. Enrique Vinicio Carrera  
**Autores:** Sebastian Abad · Daniel Cornejo  
**Objetivo M3:** Re-entrenar **solo YOLOv8s** con la receta de M1 (150 epochs, `patience=20`,
**augmentation por defecto**) pero cambiando lo único que M1/M2 no tocaron: **la distribución
de datos**. Se mezcla el dataset Roboflow v2 con un dataset **dashcam/CCTV** de dominio cercano
al video real.

---

## Contexto y motivación

M1 ("más epochs") y M2 ("augmentation dirigido") **mejoraron/movieron el test set Roboflow** pero
**REGRESARON en video real**: recall 6/8 (baseline) → 4/8 (M1) → 2/8 (M2) @ conf=0.30.
Conclusión del informe: el cuello **no es el train del detector**, es **brecha de dominio**
(dashcam/vigilancia real vs. la distribución Roboflow). No se cierra con hiperparámetros.

M3 ataca la brecha por los **datos**: añade imágenes de **dashcam/CCTV** al train para acercar
la distribución de entrenamiento a la de producción. Solo se entrena `s` (n descartado en M1).

| Parámetro | M1 | M2 | M3 (este notebook) |
|---|---|---|---|
| `epochs` / `patience` | 150 / 20 | 150 / 20 | 150 / 20 (sin cambio) |
| `batch` / `imgsz` | 16 / 640 | 16 / 640 | 16 / 640 (sin cambio) |
| Augmentation | default | **dirigido (agresivo)** | **default** (M2 fue el peor) |
| **Datos** | Roboflow v2 | Roboflow v2 | **Roboflow v2 + dashcam/CCTV** ← única palanca |

> ⚠️ **Anti-leakage:** los **8 videos WhatsApp** de test NO entran a ningún split del dataset.
> Quedan 100% held-out y solo se tocan con `tools/calibrate_threshold.py` en el repo
> `ProjectCrashIA1`. Así el 6/8 sigue siendo un test honesto.

> ⚖️ **Criterio de entrega:** la métrica que MANDA es el **recall en video real** (batir 6/8),
> no el test set Roboflow. El veredicto sale de `calibrate_threshold.py --weights` sobre los 8 videos.

---

## Contenido del notebook

| Sección | Descripción |
|---|---|
| 0 | Verificación del entorno (GPU, CUDA, RAM) |
| 1 | Configuración de rutas locales M3 |
| 2 | Dependencias e imports |
| 3 | Descarga Roboflow v2 + dashcam/CCTV **y fusión con dedup anti-leakage** |
| 4 | Re-entrenamiento YOLOv8s M3 (150 epochs, augmentation default) |
| 5 | Evaluación comparativa Baseline vs. M3 en test set |
| 6 | Visualización de curvas de entrenamiento |
| 7 | Gráfica comparativa Baseline vs. M3 |
| 8 | Exportación del modelo y artefactos |
| 9 | Resumen final y siguiente paso (validación en 8 videos) |

---
## 0. Verificación del entorno

In [ ]:
import os
# Forzar GPU 0 antes de importar torch — evita que VS Code/Jupyter inyecte CUDA_VISIBLE_DEVICES=""
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import subprocess, sys
import torch

print("=" * 55)
print("VERIFICACIÓN DE ENTORNO — CrashVision M3 (Local)")
print("=" * 55)

# GPU
try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader'],
        encoding='utf-8').strip()
    print(f"\nGPU detectada:")
    for line in gpu_info.split('\n'):
        print(f" {line}")
except FileNotFoundError:
    print("\nGPU (nvidia-smi) NO detectada.")

# RAM
import psutil
ram = psutil.virtual_memory()
print(f"\nRAM disponible: {ram.available / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB total")

# Python
print(f"\nPython: {sys.version.split()[0]}")

# CUDA via PyTorch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Dispositivo CUDA: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM total: {props.total_memory / 1e9:.1f} GB")
else:
    print("ADVERTENCIA: CUDA no detectado. M3 en CPU es inviable (horas). Revisa el kernel.")

print("\n" + "=" * 55)

---
## 1. Configuración de rutas locales M3

Mismo esquema que M1/M2 (`x\CrashVision\...`), con variables `*_M3` y carpeta `m3`. Solo `s`.

In [ ]:
import os
from pathlib import Path

# Carpeta raíz del proyecto (misma ubicación que este notebook, laptop de entrenamiento)
BASE_DIR          = Path(r"C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x")
DRIVE_BASE        = str(BASE_DIR / "CrashVision")
DRIVE_MODELS_BASE = str(BASE_DIR / "CrashVision" / "models")
DRIVE_MODELS_M3   = str(BASE_DIR / "CrashVision" / "models" / "m3")
DRIVE_RESULTS_S   = str(BASE_DIR / "CrashVision" / "results" / "yolov8s_m3")
DRIVE_COMPARE_M3  = str(BASE_DIR / "CrashVision" / "results" / "comparison_m3")

for folder in [DRIVE_MODELS_M3, DRIVE_RESULTS_S, DRIVE_COMPARE_M3]:
    os.makedirs(folder, exist_ok=True)
    print(f"Carpeta lista: {folder}")

# Verificar que el baseline 's' existe (necesario para la comparación de la Sección 5)
path_base_s = os.path.join(DRIVE_MODELS_BASE, 'best_s.pt')
if os.path.exists(path_base_s):
    print(f"Baseline encontrado: best_s.pt ({os.path.getsize(path_base_s) / 1e6:.1f} MB)")
else:
    print(f"Baseline NO encontrado: {path_base_s}")
    print("La Sección 5 necesita best_s.pt para la comparación Baseline vs M3.")

print("\nEstructura local:")
print(f"  {DRIVE_BASE}/")
print("  ├── models/")
print("  │   ├── best_s.pt        ← baseline original (NO se sobreescribe)")
print("  │   └── m3/")
print("  │       └── best_s_m3.pt ← modelo s re-entrenado con dataset de dominio")
print("  └── results/")
print("      ├── yolov8s_m3/      ← curvas y métricas s M3")
print("      └── comparison_m3/   ← tabla comparativa Baseline vs M3")

---
## 2. Dependencias e imports

In [ ]:
import ultralytics
import roboflow
print(f"Ultralytics: {ultralytics.__version__}")
print(f"Roboflow: {roboflow.__version__}")

from ultralytics import YOLO
print("YOLOv8 importado correctamente")

import os, shutil, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
print("Librerías auxiliares listas")

---
## 3. Descarga de datasets + fusión con dedup anti-leakage

**La única palanca de M3.** Se construye un único dataset YOLO (`crashvision_m3/`) mezclando:
1. **Roboflow v2** (`accident-detection-model`, misma base que M1/M2, clase única `Accident`).
2. **Fork dashcam** (`car-accident-dashcam-j7efi`, 144 imgs) — dominio cercano al video real.

> **Cómo apuntar al fork:** en Roboflow, tu fork → *Versions* → **Generate** una versión (si aún
> no la tienes) → *Download Dataset* → *YOLOv8* → pestaña *Python*. Copia `workspace(...)`,
> `project(...)` y `version(N)` EXACTOS a `DOMAIN_*` en la celda de abajo.

**Manejo de clases del fork (clave — NO es clase única):**
El fork trae 3 clases: `vehicle`, `vehicle-accident`, `vehicleaccident`. El mapeo es **por nombre**:
- cualquier clase cuyo nombre contenga **`accident`** → clase `Accident` (0).
- `vehicle` (coche normal, NO accidente) → **se descartan sus cajas**. La imagen se conserva como
  **negativo** (fondo sin accidente) → aporta negativos de dominio que ayudan a bajar los FP.

**Reglas anti-leakage aplicadas al fusionar:**
- Los splits se mezclan **por split** (train→train, valid→valid, test→test): nunca se cruza el
  valid de un dataset al train de otro.
- **Dedup global por MD5** del contenido de imagen, procesando train→valid→test: una imagen que
  ya se vio se descarta → garantiza que **ninguna imagen quede en dos splits**.
- Los **8 videos WhatsApp NO participan**: no son parte de ninguna fuente. Held-out total.

In [ ]:
from roboflow import Roboflow
from pathlib import Path
import yaml, hashlib, shutil

ROBOFLOW_API_KEY = "DwCmQ00XOYIzR0pfSFUK"

# --- Fuente 1: base Roboflow v2 (idéntica a M1/M2, clase única 'Accident') ---
BASE_WORKSPACE = "accident-detection-model"
BASE_PROJECT   = "accident-detection-model"
BASE_VERSION   = 2
BASE_API_KEY   = ROBOFLOW_API_KEY

# --- Fuente 2: fork dashcam del usuario (EDITA con el snippet de tu fork) ---
# Fork: car-accident-dashcam-j7efi (144 imgs, clases: vehicle / vehicle-accident / vehicleaccident).
# En Roboflow: tu fork -> Versions -> Generate -> Download > YOLOv8 > Python. Copia los 3 valores:

DOMAIN_WORKSPACE = "adrin-cornejo-s-workspace"
DOMAIN_PROJECT   = "car-accident-dashcam-j7efi"
DOMAIN_VERSION   = 1
                                            # <-- nº de versión generada en tu fork
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

def download(ws, proj, ver):
    ds = rf.workspace(ws).project(proj).version(ver).download("yolov8")
    loc = Path(ds.location)
    with open(loc / "data.yaml") as f:
        y = yaml.safe_load(f)
    print(f"  {proj} v{ver}: nc={y.get('nc')} clases={y.get('names')} -> {loc}")
    return loc, y

print("Descargando fuentes...")
BASE_LOC,   BASE_YAML   = download(BASE_WORKSPACE,   BASE_PROJECT,   BASE_VERSION)
DOMAIN_LOC, DOMAIN_YAML = download(DOMAIN_WORKSPACE, DOMAIN_PROJECT, DOMAIN_VERSION)

def build_class_map(yaml_dict):
    """Mapa por NOMBRE: índice original -> 0 (Accident) si el nombre contiene 'accident',
    o None (se descarta la caja) en caso contrario. Robusto ante el orden de clases."""
    names = yaml_dict.get("names", [])
    cmap = {}
    for i, name in enumerate(names):
        cmap[i] = 0 if "accident" in str(name).lower() else None
    return cmap

BASE_CMAP   = build_class_map(BASE_YAML)
DOMAIN_CMAP = build_class_map(DOMAIN_YAML)

def _describe(yaml_dict, cmap):
    names = yaml_dict.get("names", [])
    keep = [names[i] for i, t in cmap.items() if t is not None]
    drop = [names[i] for i, t in cmap.items() if t is None]
    return f"MANTENER->Accident: {keep}  |  DESCARTAR: {drop or '(ninguna)'}"

print("\nMapeo de clases (por nombre):")
print(f"  base  : {_describe(BASE_YAML, BASE_CMAP)}")
print(f"  dash  : {_describe(DOMAIN_YAML, DOMAIN_CMAP)}")
assert any(t is not None for t in DOMAIN_CMAP.values()), \
    "El fork no tiene ninguna clase 'accident': revisa DOMAIN_YAML['names']."

In [ ]:
# --- Fusión con dedup global anti-leakage + remap de clases por nombre -------
MERGED = BASE_DIR / "datasets" / "crashvision_m3"
SPLITS = ["train", "valid", "test"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Limpiar destino para una fusión reproducible
if MERGED.exists():
    shutil.rmtree(MERGED)
for sp in SPLITS:
    (MERGED / sp / "images").mkdir(parents=True, exist_ok=True)
    (MERGED / sp / "labels").mkdir(parents=True, exist_ok=True)

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()

def split_dir(loc, split):
    """Roboflow usa 'valid' pero algunos exports usan 'val'. Resuelve el nombre real."""
    for cand in ([split, "val"] if split == "valid" else [split]):
        d = loc / cand / "images"
        if d.exists():
            return loc / cand
    return None

def remap_label(src_txt, dst_txt, cmap):
    """Reescribe un .txt YOLO usando cmap {idx_original -> 0 | None}.
    Las cajas cuya clase mapea a None se DESCARTAN. Devuelve el nº de cajas conservadas
    (0 => la imagen queda como negativo/fondo, que YOLO admite con label vacío)."""
    lines_out = []
    if src_txt.exists():
        for ln in src_txt.read_text().splitlines():
            parts = ln.split()
            if len(parts) < 5:
                continue
            target = cmap.get(int(parts[0]))
            if target is None:      # p.ej. 'vehicle' -> se descarta la caja
                continue
            parts[0] = str(target)
            lines_out.append(" ".join(parts))
    dst_txt.write_text("\n".join(lines_out))
    return len(lines_out)

seen_hashes = set()   # dedup GLOBAL (a través de todos los splits y fuentes)
stats = {sp: {"kept": 0, "dup": 0, "neg": 0} for sp in SPLITS}
SOURCES = [("base", BASE_LOC, BASE_CMAP), ("dash", DOMAIN_LOC, DOMAIN_CMAP)]

# Orden importa: train -> valid -> test. 'first seen wins' => una img nunca cae en 2 splits.
for split in SPLITS:
    for tag, loc, cmap in SOURCES:
        sd = split_dir(loc, split)
        if sd is None:
            continue
        for img in sorted((sd / "images").iterdir()):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            hh = md5(img)
            if hh in seen_hashes:
                stats[split]["dup"] += 1
                continue
            seen_hashes.add(hh)
            new_stem = f"{tag}_{img.stem}"
            shutil.copy(img, MERGED / split / "images" / f"{new_stem}{img.suffix}")
            n_boxes = remap_label(sd / "labels" / f"{img.stem}.txt",
                                  MERGED / split / "labels" / f"{new_stem}.txt", cmap)
            stats[split]["kept"] += 1
            if n_boxes == 0:
                stats[split]["neg"] += 1

# data.yaml del dataset fusionado
DATA_YAML = str(MERGED / "data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump({
        "train": "train/images",
        "val":   "valid/images",
        "test":  "test/images",
        "nc": 1,
        "names": ["Accident"],
    }, f, sort_keys=False)

print("=" * 62)
print("  DATASET M3 FUSIONADO (dedup global anti-leakage + remap por nombre)")
print("=" * 62)
for sp in SPLITS:
    print(f"  {sp:6s}: {stats[sp]['kept']:5d} imgs  (negativos: {stats[sp]['neg']:4d} | "
          f"duplicados descartados: {stats[sp]['dup']})")
print(f"  Total únicas: {sum(s['kept'] for s in stats.values())}")
print(f"  data.yaml -> {DATA_YAML}")
assert sum(s['kept'] for s in stats.values()) > 0, "Fusión vacía: revisa las rutas de las fuentes."

---
## 4. Re-entrenamiento YOLOv8s M3 (150 epochs, augmentation default)

Receta heredada de M1 (150 epochs, `patience=20`, batch=16, imgsz=640, base COCO `yolov8s.pt`).
**Augmentation por DEFECTO** — M2 demostró que el augmentation agresivo fue el que más regresó
en video real. En M3 lo único que cambia respecto a M1 es **el dataset** (añadido dashcam/CCTV),
para aislar el efecto de la representatividad de dominio.

> La celda es **idempotente**: con `FORCE_RETRAIN=False`, si ya existe el run
> `runs/detect/crashvision_s_m3` no re-entrena (no sobreescribe pesos), solo lee `results.csv`.
> Pon `FORCE_RETRAIN=True` para entrenar desde cero.

In [ ]:
from ultralytics import YOLO
import torch, time

# Hiperparámetros M3 — heredados de M1 (solo 's'), augmentation DEFAULT.
EPOCHS_M3    = 150
PATIENCE_M3  = 20
BATCH_SIZE   = 16
IMG_SIZE     = 640
DEVICE       = 0  # GPU — RTX 4060 Laptop
MODEL_NAME_S = 'crashvision_s_m3'

FORCE_RETRAIN = False  # True para re-entrenar desde cero (sobreescribe el run)

run_dir = Path(f'runs/detect/{MODEL_NAME_S}')
best_pt = run_dir / 'weights' / 'best.pt'

if best_pt.exists() and not FORCE_RETRAIN:
    print("=" * 58)
    print("  RUN M3 YA EXISTE — se OMITE el entrenamiento")
    print("=" * 58)
    print(f"  Pesos: {best_pt}  ({best_pt.stat().st_size / 1e6:.1f} MB)")
    print("  FORCE_RETRAIN=False → reutiliza estos pesos. Continúa con la Sección 5.")
else:
    print("=" * 58)
    print("  RE-ENTRENAMIENTO YOLOv8s M3 — CrashVision (dataset de dominio)")
    print("=" * 58)
    print(f"  epochs={EPOCHS_M3} | patience={PATIENCE_M3} | batch={BATCH_SIZE} | imgsz={IMG_SIZE}")
    print(f"  device={DEVICE} | name={MODEL_NAME_S} | augmentation=DEFAULT")
    print("=" * 58)

    model_s_m3 = YOLO('yolov8s.pt')  # base COCO, entrenamiento desde cero (igual que M1/M2)

    t_start_s = time.time()
    results_s_m3 = model_s_m3.train(
        data=DATA_YAML,
        epochs=EPOCHS_M3,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        patience=PATIENCE_M3,
        device=DEVICE,
        name=MODEL_NAME_S,
        exist_ok=True,
        workers=2,
        cache=False,
        save=True,
        plots=True,
        verbose=True,
        # Augmentation DEFAULT: NO se pasan hsv_*/degrees/mixup/copy_paste (a propósito).
        # La única variable de M3 vs M1 es el dataset (dominio), no el augmentation.
    )
    print(f"\nYOLOv8s M3 completado en {(time.time() - t_start_s)/60:.1f} minutos")

# --- Reporte de convergencia desde results.csv ------------------------------
def _find_col(cols, needle):
    needle = needle.lower().replace(' ', '')
    for c in cols:
        if needle in c.lower().replace(' ', ''):
            return c
    return None

csv_s = run_dir / 'results.csv'
if csv_s.exists():
    df_s_check = pd.read_csv(str(csv_s))
    df_s_check.columns = df_s_check.columns.str.strip()
    n_ep = len(df_s_check)
    c50 = _find_col(df_s_check.columns, 'map50(b)')
    c95 = _find_col(df_s_check.columns, 'map50-95(b)')
    best_line = ''
    if c50 and c95:
        fitness = 0.1 * df_s_check[c50] + 0.9 * df_s_check[c95]
        best_i = int(fitness.idxmax())
        best_epoch = int(df_s_check.iloc[best_i].get('epoch', best_i + 1))
        best_line = (f"  Mejor epoch ≈ {best_epoch} "
                     f"(mAP50={df_s_check[c50].iloc[best_i]:.3f}, "
                     f"mAP50-95={df_s_check[c95].iloc[best_i]:.3f}) → es el que quedó en best.pt")
    if n_ep < EPOCHS_M3:
        print(f"\nEarly stopping activado en epoch {n_ep} / {EPOCHS_M3}  ← modelo convergió")
    else:
        print(f"\nEl modelo completó las {EPOCHS_M3} epochs SIN early stopping.")
    if best_line:
        print(best_line)

---
## 5. Evaluación comparativa Baseline vs. M3 (test set, conf=0.30)

Solo `s`: `best_s.pt` (baseline) vs. M3, sobre el **test set fusionado**. **Recordatorio:** el
test set NO es el criterio de entrega — el veredicto real es el recall en los 8 videos (Sección 9).

In [ ]:
from ultralytics import YOLO

CONF_EVAL = 0.30  # threshold óptimo — Experimento 1

models_eval = {
    'YOLOv8s Baseline': f'{DRIVE_MODELS_BASE}/best_s.pt',
    'YOLOv8s M3':       f'runs/detect/{MODEL_NAME_S}/weights/best.pt',
}

for name, path in models_eval.items():
    if os.path.exists(path):
        print(f"{name:<22} → {os.path.getsize(path) / 1e6:.1f} MB")
    else:
        print(f"NO encontrado: {name} → {path}")

print(f"\nEvaluando en test set (conf={CONF_EVAL})...")
print("-" * 65)

comparison_results = []

for model_name, model_path in models_eval.items():
    if not os.path.exists(model_path):
        print(f"Saltando {model_name} (archivo no encontrado)")
        continue

    print(f"  Evaluando {model_name}...", end=' ', flush=True)
    m_obj = YOLO(model_path)
    m = m_obj.val(
        data=DATA_YAML,
        split='test',
        conf=CONF_EVAL,
        iou=0.50,
        verbose=False,
    )
    precision = m.box.mp
    recall    = m.box.mr
    map50     = m.box.map50
    map_full  = m.box.map
    f1        = 2 * (precision * recall) / (precision + recall + 1e-9)
    size_mb   = os.path.getsize(model_path) / 1e6

    comparison_results.append({
        'Modelo':     model_name,
        'Precision':  round(precision, 4),
        'Recall':     round(recall, 4),
        'F1-score':   round(f1, 4),
        'mAP@50':     round(map50, 4),
        'mAP@50:95':  round(map_full, 4),
        'Tamaño(MB)': round(size_mb, 1),
    })
    print(f"P={precision:.3f} | R={recall:.3f} | F1={f1:.3f} | mAP@50={map50:.3f}")

df_comparison = pd.DataFrame(comparison_results)

print("\n" + "=" * 75)
print("  COMPARACIÓN BASELINE vs M3 (conf=0.30, test set)")
print("=" * 75)
print(df_comparison.to_string(index=False))

print("\n── Análisis de mejora (test set) ──")
base_rows = df_comparison[df_comparison['Modelo'] == 'YOLOv8s Baseline']
m3_rows   = df_comparison[df_comparison['Modelo'] == 'YOLOv8s M3']
if not base_rows.empty and not m3_rows.empty:
    base_row = base_rows.iloc[0]
    m3_row   = m3_rows.iloc[0]
    sign = lambda v: f"+{v:.4f}" if v >= 0 else f"{v:.4f}"
    delta_f1    = m3_row['F1-score'] - base_row['F1-score']
    delta_map50 = m3_row['mAP@50']   - base_row['mAP@50']
    delta_rec   = m3_row['Recall']   - base_row['Recall']
    print(f"  YOLOv8s M3      : ΔF1={sign(delta_f1)} | ΔmAP@50={sign(delta_map50)} | ΔRecall={sign(delta_rec)}")
    print("  (Nota: la métrica que decide la entrega es el recall en VIDEO real, no ésta.)")

csv_out = f'{DRIVE_COMPARE_M3}/m3_vs_baseline.csv'
df_comparison.to_csv(csv_out, index=False)
print(f"\nm3_vs_baseline.csv guardado en {csv_out}")

---
## 6. Curvas de entrenamiento — YOLOv8s M3

In [ ]:
def safe_plot(ax, df, col_name, title, color, ylabel=''):
    """Grafica una columna si existe, con matching parcial de nombre."""
    match = [c for c in df.columns if col_name.lower() in c.lower()]
    if match:
        epoch_col = 'epoch' if 'epoch' in df.columns else df.columns[0]
        ax.plot(df[epoch_col], df[match[0]], color=color, linewidth=1.8)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel or title)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(left=0)
    else:
        ax.text(0.5, 0.5, f'No disponible\n({col_name})', ha='center', va='center',
                transform=ax.transAxes, color='gray')
        ax.set_title(title, fontsize=10)

csv_s_m3 = f'runs/detect/{MODEL_NAME_S}/results.csv'
if os.path.exists(csv_s_m3):
    df_s = pd.read_csv(csv_s_m3)
    df_s.columns = df_s.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f'Curvas de Entrenamiento — YOLOv8s M3 ({len(df_s)} epochs)', fontsize=14, fontweight='bold')

    safe_plot(axes[0,0], df_s, 'train/box_loss',   'Box Loss (train)',    '#E74C3C', 'Loss')
    safe_plot(axes[0,1], df_s, 'train/cls_loss',   'Class Loss (train)',  '#E67E22', 'Loss')
    safe_plot(axes[0,2], df_s, 'train/dfl_loss',   'DFL Loss (train)',    '#F39C12', 'Loss')
    safe_plot(axes[1,0], df_s, 'metrics/mAP50',    'mAP@50 (val)',        '#27AE60', 'mAP')
    safe_plot(axes[1,1], df_s, 'metrics/precision','Precision (val)',      '#2980B9', 'Precision')
    safe_plot(axes[1,2], df_s, 'metrics/recall',   'Recall (val)',         '#8E44AD', 'Recall')

    plt.tight_layout()
    plt.savefig(f'{DRIVE_RESULTS_S}/training_curves_m3.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Curvas YOLOv8s M3 guardadas en", DRIVE_RESULTS_S)
else:
    print(f"results.csv no encontrado: {csv_s_m3}")

---
## 7. Gráfica comparativa Baseline vs. M3

In [ ]:
if len(df_comparison) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 6))
    fig.suptitle('Baseline vs M3 — CrashVision (conf=0.30, test set)', fontsize=14, fontweight='bold')

    model_labels = df_comparison['Modelo'].tolist()
    x = np.arange(len(model_labels))
    colors = ['#95A5A6', '#16A085'][:len(model_labels)]  # gris=baseline, verde=M3

    ax1 = axes[0]
    w = 0.3
    f1_vals    = df_comparison['F1-score'].tolist()
    map50_vals = df_comparison['mAP@50'].tolist()
    b1 = ax1.bar(x - w/2, f1_vals,    w, label='F1-score', color='#2980B9', alpha=0.85)
    b2 = ax1.bar(x + w/2, map50_vals, w, label='mAP@50',   color='#27AE60', alpha=0.85)
    for bar, v in list(zip(b1, f1_vals)) + list(zip(b2, map50_vals)):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax1.set_xticks(x); ax1.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=8)
    ax1.set_ylim(0, 1.05); ax1.set_title('F1-score y mAP@50'); ax1.set_ylabel('Score')
    ax1.legend(); ax1.grid(axis='y', alpha=0.3)

    ax2 = axes[1]
    prec_vals = df_comparison['Precision'].tolist()
    rec_vals  = df_comparison['Recall'].tolist()
    b3 = ax2.bar(x - w/2, prec_vals, w, label='Precision', color='#E67E22', alpha=0.85)
    b4 = ax2.bar(x + w/2, rec_vals,  w, label='Recall',    color='#8E44AD', alpha=0.85)
    for bar, v in list(zip(b3, prec_vals)) + list(zip(b4, rec_vals)):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax2.set_xticks(x); ax2.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=8)
    ax2.set_ylim(0, 1.05); ax2.set_title('Precision y Recall'); ax2.set_ylabel('Score')
    ax2.legend(); ax2.grid(axis='y', alpha=0.3)

    ax3 = axes[2]
    map_full_vals = df_comparison['mAP@50:95'].tolist()
    bars = ax3.bar(model_labels, map_full_vals, color=colors, alpha=0.85, width=0.5)
    for bar, v in zip(bars, map_full_vals):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax3.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=8)
    ax3.set_ylim(0, 0.7); ax3.set_title('mAP@50:95'); ax3.set_ylabel('mAP@50:95')
    ax3.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{DRIVE_COMPARE_M3}/m3_vs_baseline_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Gráfica comparativa guardada en", DRIVE_COMPARE_M3)

---
## 8. Exportación del modelo y artefactos

Copia `best_s_m3.pt`/`last_s_m3.pt` a `models/m3/` y los artefactos de `runs/` a `results/yolov8s_m3/`.

In [ ]:
import shutil, os

best_src = f'runs/detect/{MODEL_NAME_S}/weights/best.pt'
best_dst = os.path.join(DRIVE_MODELS_M3, 'best_s_m3.pt')
last_src = f'runs/detect/{MODEL_NAME_S}/weights/last.pt'
last_dst = os.path.join(DRIVE_MODELS_M3, 'last_s_m3.pt')

print("=" * 58)
print("EXPORTANDO MODELO M3 A CARPETA LOCAL")
print("=" * 58)

if not os.path.exists(best_src):
    print(f"YOLOv8s M3: best.pt no encontrado en {best_src}")
else:
    shutil.copy(best_src, best_dst)
    if os.path.exists(last_src):
        shutil.copy(last_src, last_dst)

    size_mb = os.path.getsize(best_dst) / 1e6
    print(f"\nYOLOv8s M3")
    print(f"{best_dst} ({size_mb:.1f} MB)")

    results_dir = Path(f'runs/detect/{MODEL_NAME_S}')
    artifacts   = [
        'results.csv',
        'confusion_matrix.png',
        'BoxPR_curve.png', 'PR_curve.png',
        'BoxF1_curve.png', 'F1_curve.png',
        'results.png',
    ]
    for fname in artifacts:
        src = results_dir / fname
        if src.exists():
            shutil.copy(str(src), os.path.join(DRIVE_RESULTS_S, fname))
            print(f"  {fname}")

print("\n" + "=" * 58)
print("  SIGUIENTE PASO — validar en VIDEO real (repo ProjectCrashIA1):")
print("  1) Copia best_s_m3.pt a  ProjectCrashIA1/models/best_s_m3.pt")
print("  2) venv/Scripts/python tools/calibrate_threshold.py \\")
print("       --weights models/best_s_m3.pt --labels tools/labels.csv")
print("  La entrega se decide por el recall en los 8 videos (batir 6/8).")

---
## 9. Resumen final y siguiente paso

In [ ]:
print("\n" + "=" * 68)
print("  RESUMEN FINAL — CrashVision M3")
print("=" * 68)

print("\nEarly stopping:")
if os.path.exists(csv_s_m3):
    df_tmp = pd.read_csv(csv_s_m3)
    n_ep = len(df_tmp)
    status = f"paró en epoch {n_ep} (convergió)" if n_ep < EPOCHS_M3 else f"completó {n_ep} epochs sin parar"
    print(f"   YOLOv8s M3        : {status}")

if not df_comparison.empty:
    print("\nComparación Baseline vs M3 (conf=0.30, test set):")
    print(df_comparison[['Modelo','Precision','Recall','F1-score','mAP@50','mAP@50:95']].to_string(index=False))

print("\n⚖️  DECISIÓN DE ENTREGA — se toma en VIDEO real, no aquí:")
print("   En el repo ProjectCrashIA1:")
print("     venv/Scripts/python tools/calibrate_threshold.py --weights models/best_s_m3.pt \\")
print("        --labels tools/labels.csv")
print("   Baseline a batir: recall 6/8 (75%). M1 regresó a 4/8, M2 a 2/8.")
print("   - Si M3 > 6/8  → nuevo ganador; recién ahí promover a config.py + TRAINING_METRICS.")
print("   - Si M3 ≤ 6/8  → mantener best_s.pt; refuerza que el train no cierra la brecha de")
print("                    dominio y se consolida el post-proceso temporal + baseline como prod.")

print("\nArchivos generados localmente:")
for f in [
    os.path.join(DRIVE_MODELS_M3, 'best_s_m3.pt'),
    os.path.join(DRIVE_RESULTS_S, 'results.csv'),
    os.path.join(DRIVE_RESULTS_S, 'training_curves_m3.png'),
    os.path.join(DRIVE_COMPARE_M3, 'm3_vs_baseline.csv'),
    os.path.join(DRIVE_COMPARE_M3, 'm3_vs_baseline_comparison.png'),
]:
    print(f"  {f}")
print("\n" + "=" * 68)